# 13.08 NVIDIA — NIM 엔드포인트 카탈로그

`langchain-nvidia-ai-endpoints` 는 **NVIDIA NIM**(NVIDIA Inference Microservice) 으로 노출되는 chat / embedding / reranker 를 단일 패키지로 묶는다. 두 가지 호스팅 모드 — **NVIDIA 클라우드(`build.nvidia.com`)** 와 **자체 호스팅 NIM 컨테이너** — 가 같은 클래스 인터페이스로 동작한다.

## 학습 목표

- NVIDIA 패키지가 노출하는 3 클래스(`ChatNVIDIA` · `NVIDIAEmbeddings` · `NVIDIARerank`) 를 한눈에 본다
- 클라우드 vs. 자체 호스팅(NIM 컨테이너) 의 선택 기준을 안다
- function calling · multimodal 지원 모델 ID 를 식별한다

## 언제 이 공급자를 고르나

- **자체 GPU 인프라**가 있고 모델을 사내에 두고 싶을 때(NIM 컨테이너 self-host)
- 오픈 가중치 모델(Llama, Mixtral 등)을 NVIDIA 가 최적화한 엔진(TensorRT-LLM)으로 돌리고 싶을 때
- chat + embedding + rerank 를 **한 키 한 패키지**로 통일하고 싶을 때
- 다국어 임베딩(`nvidia/nv-embed-v1` · `nvidia/nv-embedqa-mistral-7b-v2`) · rerank(`nvidia/nv-rerankqa-mistral-4b-v3`) 가 필요할 때

## 13.08.1 환경 설정

필요 키는 `NVIDIA_API_KEY` 한 개. https://build.nvidia.com 에서 발급한다. 자체 호스팅 NIM 을 쓸 때는 `base_url` 을 컨테이너 주소(`http://nim:8000/v1`)로 바꾸고 키는 임의값(보통 'NA') 사용.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
assert os.environ["NVIDIA_API_KEY"], "NVIDIA_API_KEY 가 .env 에 필요합니다"

from langchain_nvidia_ai_endpoints import ChatNVIDIA

probe = ChatNVIDIA(model="meta/llama-3.1-8b-instruct", max_tokens=64)
print(probe.invoke("ping").content[:80])

## 13.08.2 공급자 제품군 전체 지도

| 영역 | 심볼 / 모델 ID 예 | 비고 / 링크 |
|------|--------------------|-------------|
| Chat | `ChatNVIDIA(model="meta/llama-3.1-70b-instruct")` | function calling 지원 |
| Chat (vision) | `ChatNVIDIA(model="meta/llama-3.2-90b-vision-instruct")` | 이미지 입력 지원 |
| Embeddings | `NVIDIAEmbeddings(model="nvidia/nv-embed-v1")` | 4096-dim, 다국어 |
| Reranker | `NVIDIARerank(model="nvidia/nv-rerankqa-mistral-4b-v3")` | 컨텍스트 후처리 |
| Self-hosted NIM | `ChatNVIDIA(base_url="http://nim:8000/v1", model=...)` | 같은 클래스, 다른 base_url |
| 카테고리 cross-link | `01_chat_models/` 의 NVIDIA 섹션 (chat) · `02_embeddings/` (embed) · `06_rerank/` (rerank) | 깊은 구현 |

**중요**: 같은 클래스가 클라우드와 self-host 양쪽을 모두 처리한다 — 차이는 `base_url` 한 줄.

## 13.08.3 기본 사용 — chat

OpenAI/Groq 와 같은 v1 표준 인터페이스. 모델 ID 만 `meta/llama-3.1-...` 형태로 NVIDIA 카탈로그 명세를 따른다.

In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

llm = ChatNVIDIA(model="meta/llama-3.1-70b-instruct", temperature=0, max_tokens=256)
msg = llm.invoke("NIM 컨테이너의 핵심 가치를 한 문장으로 한국어로 설명해라")
print(msg.content)

## 13.08.4 공급자 특화 기능

### NIM 컨테이너 자체 호스팅

NVIDIA 가 모델별로 제공하는 **컨테이너 이미지**(예: `nvcr.io/nim/meta/llama-3.1-8b-instruct:latest`) 를 사내 GPU 클러스터에 띄우면 같은 LangChain 클래스로 호출할 수 있다. 데이터 주권/HIPAA/보안 컴플라이언스 환경에서 핵심 활용처.

```bash
docker run --rm --gpus all -p 8000:8000 \
  -e NGC_API_KEY=$NGC_API_KEY \
  nvcr.io/nim/meta/llama-3.1-8b-instruct:latest
```

```python
from langchain_nvidia_ai_endpoints import ChatNVIDIA
llm = ChatNVIDIA(
    base_url="http://localhost:8000/v1",
    model="meta/llama-3.1-8b-instruct",
    api_key="NA",  # self-host 에서는 키 검증 없음
)
```

### function calling

Llama 3.1 70B/8B 와 Mixtral 모델은 OpenAI 스타일 tool 호출을 지원한다. `bind_tools(...)` 동일.

In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain.tools import tool

@tool
def get_temperature(city: str) -> float:
    """도시의 현재 기온을 섭씨로 반환한다."""
    return 21.5

llm = ChatNVIDIA(model="meta/llama-3.1-70b-instruct", temperature=0).bind_tools([get_temperature])
out = llm.invoke("서울 기온 알려줘")
print(out.tool_calls)

### Embeddings & Reranker

한 키로 검색 파이프라인 전 구간을 NVIDIA 로 통일할 수 있다.

```python
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings, NVIDIARerank

embed = NVIDIAEmbeddings(model="nvidia/nv-embed-v1")
vec = embed.embed_query("langchain 이란")  # 4096-dim

rerank = NVIDIARerank(model="nvidia/nv-rerankqa-mistral-4b-v3", top_n=3)
ranked = rerank.compress_documents(documents=[...], query="...")
```

## 13.08.5 가격·한도·리전

| 항목 | 값 (2025 공개치) |
|------|------------------|
| 클라우드 엔드포인트 | `https://integrate.api.nvidia.com/v1` (단일 리전) |
| 무료 크레딧 | 회원가입 시 일정량의 free request credits 제공 |
| 유료 가격 | 모델별로 1k req 단위 또는 1M tok 단위. 카탈로그 페이지에 표기 |
| Self-host NIM | NIM 라이선스 + 사내 GPU 비용. NVIDIA AI Enterprise 구독 필요 |
| Rate limit | 클라우드는 모델별 RPM/TPM, self-host 는 GPU 메모리/스루풋이 한도 |
| 컨텍스트 한도 | 모델별 — Llama 3.1 은 128k, Llama 3.2 vision 은 텍스트 128k+이미지 |

최신 단가는 https://build.nvidia.com 의 모델 카탈로그 카드, NIM 라이선스는 NVIDIA AI Enterprise 영업 채널.

## 핵심 정리

- 패키지 한 개에 chat · embedding · rerank 3 클래스가 들어있고, 자체 호스팅과 클라우드가 같은 클래스로 처리됨
- 차별점은 **TensorRT-LLM 최적화 + NIM 컨테이너 self-host 경로**
- function calling · multimodal(vision) · 다국어 embedding 모두 OK
- 데이터 주권/컴플라이언스가 중요한 환경에서 1순위 후보

## 다음

- chat 깊은 구현: `07_integration/01_chat_models/` 의 NVIDIA 섹션
- 다음 공급자 카탈로그: `09_ollama.ipynb`

## 참고

- 공식 통합 페이지: https://docs.langchain.com/oss/python/integrations/providers/nvidia
- NVIDIA build 카탈로그: https://build.nvidia.com
- NIM 컨테이너 가이드: https://docs.nvidia.com/nim/